# PA1 - Inferência Rápida e Interativa

Este notebook permite que você carregue qualquer imagem local e rode a predição completa do nosso modelo final (Trilha A - 3 Classes + Watershed) para segmentar as instâncias de núcleos.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Garante que o pacote pa1 é acessível
repo_root = Path(".").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pa1.models import UNet
from pa1.postprocessing import watershed_to_instances
from pa1.utils import get_device

# 1. Configurações Iniciais
device = get_device()
print(f"\nInicializando na {device}...")

# 2. Carregando o Modelo (Baseline Trilha A)
ckpt_path = Path("pa1/outputs/checkpoints/parte2_baseline_unet.pt")
if not ckpt_path.exists():
    raise FileNotFoundError(f"Checkpoint não encontrado em {ckpt_path}. Rode o treino da parte 2 primeiro.")

model = UNet(in_channels=3, out_channels=3).to(device)
model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
model.eval()
print(f"Modelo carregado com sucesso: {ckpt_path.name}")

### Selecione a Imagem

In [ ]:
# Escolha o caminho da imagem aqui (pode ser uma do DSB2018 ou outra externa)
IMAGE_PATH = "pa1/data/stage1_train/0a7d30b252359a10fd298b638b90cb9ada3acced4e0c0e5a3692013f432ee4e9/images/0a7d30b252359a10fd298b638b90cb9ada3acced4e0c0e5a3692013f432ee4e9.png"

# Lê a imagem e converte para RGB
img_bgr = cv2.imread(IMAGE_PATH)
if img_bgr is None:
    img_pil = Image.open(IMAGE_PATH).convert("RGB")
    img_rgb = np.array(img_pil)
else:
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

print(f"Imagem original carregada: {img_rgb.shape} ({img_rgb.dtype})")

### Pré-processamento, Inferência e Decodificação

In [ ]:
# 1. Normalização ImageNet (usada no treino)
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
img_f = img_rgb.astype(np.float32) / 255.0
img_norm = (img_f - mean) / std

# 2. Tensorização
tensor = torch.from_numpy(img_norm.transpose(2, 0, 1)).unsqueeze(0).float().to(device)

# 3. Inferência (Forward Pass)
with torch.no_grad():
    logits = model(tensor)
    probs = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()

# 4. Watershed
pred_inst = watershed_to_instances(probs, interior_channel=1, marker_threshold=0.6, min_area=15)
num_cells = len(np.unique(pred_inst)) - 1

### Visualização dos Resultados

In [ ]:
# Configura colormap (fundo preto, resto colorido aleatório)
np.random.seed(42)
colors = np.random.rand(10000, 3)
colors[0] = [0, 0, 0] 
cmap = mcolors.ListedColormap(colors)

# Plot
fig, axs = plt.subplots(1, 3, figsize=(16, 5))

axs[0].imshow(img_rgb)
axs[0].set_title("Imagem Original")
axs[0].axis("off")

axs[1].imshow(probs[2], cmap="magma")
axs[1].set_title("Probabilidade de Fronteira (Classe 2)")
axs[1].axis("off")

axs[2].imshow(pred_inst, cmap=cmap, interpolation="nearest")
axs[2].set_title(f"Instâncias Preditas (Contagem: {num_cells})")
axs[2].axis("off")

plt.tight_layout()
plt.show()